# Session 8: LAN Switching Technologies

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Hamnakhan123/Cybersecurity-and-Digital-Forensics/blob/main/term-1/03-computer-networks/code/session-08-lan-switching.ipynb)

*Cybersecurity and Digital Forensics* · Prepared by **Hamna Zahid Khan**, Junior Lecturer

Run each cell from top to bottom (Shift+Enter). Every example is self-contained and safe to run in a lab or on Colab.

## `session-08_dot1q.py`

In [1]:
"""Session 8 - Build and read an IEEE 802.1Q (VLAN) tagged Ethernet header.

A trunk link carries frames for many VLANs. The 4-byte tag sits between the
source MAC and the EtherType: TPID 0x8100 + 3-bit priority + 1-bit DEI + 12-bit VLAN ID.
Only bytes in memory are built here; nothing is sent.
Run:  python session-08_dot1q.py
"""
import struct


def mac_bytes(mac):
    return bytes(int(x, 16) for x in mac.split(":"))


def build_frame(dst, src, vlan=None, prio=0, ethertype=0x0800, payload=b"IPv4..."):
    hdr = mac_bytes(dst) + mac_bytes(src)
    if vlan is not None:
        if not 1 <= vlan <= 4094:
            raise ValueError("VLAN ID must be 1-4094 (0 and 4095 are reserved)")
        tci = (prio << 13) | (0 << 12) | vlan
        hdr += struct.pack("!HH", 0x8100, tci)
    return hdr + struct.pack("!H", ethertype) + payload


def parse(frame):
    dst = ":".join(f"{b:02x}" for b in frame[0:6])
    src = ":".join(f"{b:02x}" for b in frame[6:12])
    (tpid,) = struct.unpack("!H", frame[12:14])
    if tpid == 0x8100:
        (tci,) = struct.unpack("!H", frame[14:16])
        (etype,) = struct.unpack("!H", frame[16:18])
        return dict(dst=dst, src=src, tagged=True, prio=tci >> 13, vlan=tci & 0x0FFF, etype=hex(etype))
    return dict(dst=dst, src=src, tagged=False, vlan="native/access", etype=hex(tpid))


if __name__ == "__main__":
    pc = "aa:bb:cc:00:00:10"; gw = "aa:bb:cc:00:00:01"
    untagged = build_frame(gw, pc)
    voice = build_frame(gw, pc, vlan=20, prio=5)
    data = build_frame(gw, pc, vlan=10)
    for name, f in [("access port (untagged)", untagged), ("trunk, VLAN 10 data", data),
                    ("trunk, VLAN 20 voice", voice)]:
        print(f"{name:<24} {len(f):>2} bytes  hex: {f[:18].hex(' ')}")
        print(f"{'':<24} parsed -> {parse(f)}")
    print("\nTag adds", len(data) - len(untagged), "bytes; 12-bit VLAN ID gives", 2 ** 12 - 2, "usable VLANs")
    try:
        build_frame(gw, pc, vlan=5000)
    except ValueError as e:
        print("Rejected:", e)

access port (untagged)   21 bytes  hex: aa bb cc 00 00 01 aa bb cc 00 00 10 08 00 49 50 76 34
                         parsed -> {'dst': 'aa:bb:cc:00:00:01', 'src': 'aa:bb:cc:00:00:10', 'tagged': False, 'vlan': 'native/access', 'etype': '0x800'}
trunk, VLAN 10 data      25 bytes  hex: aa bb cc 00 00 01 aa bb cc 00 00 10 81 00 00 0a 08 00
                         parsed -> {'dst': 'aa:bb:cc:00:00:01', 'src': 'aa:bb:cc:00:00:10', 'tagged': True, 'prio': 0, 'vlan': 10, 'etype': '0x800'}
trunk, VLAN 20 voice     25 bytes  hex: aa bb cc 00 00 01 aa bb cc 00 00 10 81 00 a0 14 08 00
                         parsed -> {'dst': 'aa:bb:cc:00:00:01', 'src': 'aa:bb:cc:00:00:10', 'tagged': True, 'prio': 5, 'vlan': 20, 'etype': '0x800'}

Tag adds 4 bytes; 12-bit VLAN ID gives 4094 usable VLANs
Rejected: VLAN ID must be 1-4094 (0 and 4095 are reserved)


## `session-08_arp_watch.py`

In [2]:
"""Session 8 - ARP cache behaviour and a defensive ARP-spoofing detector.

ARP maps an IPv4 address to a MAC address on the local network. Because ARP
has no authentication, an attacker on the LAN can send fake replies. A monitor
(like the classic `arpwatch` tool) flags when an IP suddenly changes MAC.
The events below are a recorded lab capture, not live traffic.
Run:  python session-08_arp_watch.py
"""

EVENTS = [  # (time, type, sender IP, sender MAC)
    ("10:00:01", "reply",  "192.168.1.1",  "00:11:22:33:44:01"),   # gateway
    ("10:00:03", "reply",  "192.168.1.20", "00:11:22:33:44:20"),
    ("10:00:09", "reply",  "192.168.1.30", "00:11:22:33:44:30"),
    ("10:05:00", "garp",   "192.168.1.20", "00:11:22:33:44:20"),   # normal refresh
    ("10:07:12", "reply",  "192.168.1.1",  "de:ad:be:ef:00:66"),   # gateway IP, new MAC!
    ("10:07:13", "reply",  "192.168.1.30", "de:ad:be:ef:00:66"),   # same MAC claims a 2nd IP
    ("10:07:14", "reply",  "192.168.1.1",  "de:ad:be:ef:00:66"),
]
CRITICAL = {"192.168.1.1": "default gateway"}


def watch(events):
    table, alerts, mac_to_ips, seen = {}, [], {}, set()
    for t, kind, ip, mac in events:
        old = table.get(ip)
        if old and old != mac:
            sev = "CRITICAL" if ip in CRITICAL else "WARNING"
            alerts.append(f"{t} {sev}: {ip} changed {old} -> {mac}")
        table[ip] = mac
        mac_to_ips.setdefault(mac, set()).add(ip)
        key = (mac, frozenset(mac_to_ips[mac]))
        if len(mac_to_ips[mac]) > 1 and key not in seen:
            seen.add(key)
            alerts.append(f"{t} WARNING: MAC {mac} now claims {sorted(mac_to_ips[mac])}")
    return table, alerts


if __name__ == "__main__":
    table, alerts = watch(EVENTS)
    print("Final ARP cache (what victims now believe):")
    for ip, mac in table.items():
        print(f"  {ip:<14} -> {mac}")
    print("\nAlerts:")
    for a in alerts:
        print(" ", a)
    print("\nDefences: Dynamic ARP Inspection + DHCP snooping on switches, "
          "static ARP for the gateway on critical hosts, encrypt traffic (TLS/VPN).")

Final ARP cache (what victims now believe):
  192.168.1.1    -> de:ad:be:ef:00:66
  192.168.1.20   -> 00:11:22:33:44:20
  192.168.1.30   -> de:ad:be:ef:00:66

Alerts:
  10:07:12 CRITICAL: 192.168.1.1 changed 00:11:22:33:44:01 -> de:ad:be:ef:00:66
  10:07:13 WARNING: 192.168.1.30 changed 00:11:22:33:44:30 -> de:ad:be:ef:00:66
  10:07:13 WARNING: MAC de:ad:be:ef:00:66 now claims ['192.168.1.1', '192.168.1.30']

Defences: Dynamic ARP Inspection + DHCP snooping on switches, static ARP for the gateway on critical hosts, encrypt traffic (TLS/VPN).
